# Module 8 - Lab 1: Build an Agentic IT Operations Assistant with LangChain Built-in Tools

**Module:** 8 - Advanced LLM Systems &nbsp;|&nbsp; **Scenario:** IT Infrastructure Operations &nbsp;|&nbsp; **Main artifact:** this notebook

In this lab you build an **AI agent** for an IT Operations team. You type a request in plain English. The agent decides which tools to use, runs them, reads what they return, and then writes a response based on those results.

The agent uses three tool categories from LangChain's built-in tools:

1. **File Management** - list, search and read files in `operations_workspace/`, and write new report files.
2. **Python REPL** - run small Python calculations on the data.
3. **Web/Search** - look up generic public technical information (DuckDuckGo, no API key needed).

**Learning objectives**

1. Explain what an AI agent is and how it differs from a normal LLM application.
2. Explain how a LangChain agent selects and executes tools.
3. Use the File Management, Python REPL and Web/Search tools.
4. Observe the agent's tool selection in the verbose execution trace.
5. Run four IT Operations use cases that combine several tools.
6. Recognise basic tool errors and agent limitations.

**How to run this notebook:** run the cells from top to bottom (Run All also works). Sections 1 to 6 work without an API key. Sections 7 to 10 call the OpenAI API, so `OPENAI_API_KEY` must be set in `.env` (see the README). Each agent run costs a small amount of API credit.

## Section 1 - Introduction to AI Agents

**What is an AI agent?** An AI agent is a program that uses a language model (LLM) to *decide what to do next*. Instead of only writing an answer, it can call **tools** (functions it is allowed to run), read the results, and continue until it can answer.

**What is agentic AI?** Agentic AI is the pattern where an LLM works in steps: think, act, observe, and repeat. It is useful when the answer depends on information the model does not have, such as today's log file or the current server metrics.

**LLM application vs. AI agent**

| | Normal LLM application | AI agent |
|---|---|---|
| Input | A user question | A user request |
| Source of facts | The model's training data and the prompt | Tools: files, calculations, web search |
| Number of steps | One call, one answer | Several steps, decided by the agent |
| Checks the data? | No | Yes - it reads the files before it answers |
| Example | "What does CPU utilization mean?" | "Find the critical errors in application.log and count them." |

**Key terms**

- **Tool:** a function with a name and a description, such as `read_file`. The description tells the LLM when the tool is useful.
- **Tool calling:** the LLM returns a structured request such as "call `read_file` with `file_path = application.log`". The program runs the function and sends the result back.
- **Tool observation:** the output the tool returned. The agent reads it before it decides the next step.
- **Agent execution loop:** the cycle of *think, call tool, observe, think again*. It stops when the agent gives a final answer or reaches the iteration limit.

**How does the agent select a tool?** The LLM reads the user request and the tool descriptions, then picks the tool whose description fits the task. Nothing is hard-coded. This is why the tool descriptions matter (Section 7).

```mermaid
flowchart LR
    A[User Request] --> B[Agent]
    B --> C[Tool Selection]
    C --> D[Tool Execution]
    D --> E[Tool Output]
    E --> B
    B --> F[Final Response]
```

The agent decides which tool to invoke based on the task. It may call one tool, several tools, or none.

## Section 2 - Understanding the Lab Environment

- **Existing Python environment.** This lab uses the Python interpreter that is already installed on your computer, the same one used for Module 7. Do **not** create a virtual environment or install another Python. The notebook kernel must be the same interpreter where the packages are installed.
- **LangChain packages.** The core versions are the ones used in Module 7 Lab 1 and Lab 2: `langchain` 1.3.14, `langchain-core` 1.5.3, `langchain-community` 0.4.2 and `langchain-openai` 1.4.2. Three supporting packages are added for this lab: `langchain-classic` (agent API), `langchain-experimental` (Python REPL) and `ddgs` (DuckDuckGo search backend).
- **OpenAI API configuration.** The LLM is `gpt-4o-mini` through `ChatOpenAI`, the same model as Module 7. The key is read from a `.env` file with `python-dotenv`. The notebook checks that the key exists but never prints it.
- **Operations workspace.** `operations_workspace/` holds the three sample files. The agent may read them and write new reports there, and nothing else.
- **Three tool categories.** File Management, Python REPL and Web/Search. They are explained in Sections 4, 5 and 6.

In [ ]:
# Standard library and environment helpers
import builtins
import csv
import hashlib
import os
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

from dotenv import load_dotenv

# LangChain model, tools and agent components
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain_community.agent_toolkits import FileManagementToolkit
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.tools.file_management.write import WriteFileTool
from langchain_experimental.tools import PythonREPLTool
from langchain_experimental.utilities.python import PythonREPL
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent

print("Libraries imported.\n")
print("Python executable:", sys.executable)
print("Python version   :", sys.version.split()[0])
print()
for package in ["langchain", "langchain-core", "langchain-community", "langchain-openai",
                "langchain-classic", "langchain-experimental", "ddgs", "python-dotenv"]:
    try:
        print(f"{package:22} {version(package)}")
    except PackageNotFoundError:
        print(f"{package:22} NOT INSTALLED")

In [ ]:
# Read OPENAI_API_KEY from the .env file in this folder (if the file exists)
load_dotenv()

# Check that a real key is available. The key itself is never printed.
api_key = os.getenv("OPENAI_API_KEY", "")
api_key_found = bool(api_key) and api_key != "your_openai_api_key_here"
print("OPENAI_API_KEY available:", api_key_found)
if not api_key_found:
    print("WARNING: no key found. Sections 1 to 6 still run. Sections 7 to 10 need a key in .env.")

# Lab settings: change these values to experiment
WORKSPACE = (Path.cwd() / "operations_workspace").resolve()  # the ONLY folder the agent may use
CHAT_MODEL = "gpt-4o-mini"  # the LLM that drives the agent (same model as Module 7)
MAX_ITERATIONS = 8          # safety limit: maximum reasoning and tool steps per request

assert WORKSPACE.is_dir(), f"operations_workspace not found at {WORKSPACE}. Open the notebook from the lab folder."
print("\nWorkspace :", WORKSPACE)
print("Chat model:", CHAT_MODEL)
print("Max steps :", MAX_ITERATIONS)

## Section 3 - Inspect the Operations Workspace

The workspace holds three sample files. They are the operational sources the agent will work with.

| File | What it contains | Used for |
|---|---|---|
| `application.log` | 32 log lines from the `orders-api` application on 19 Nov 2026: startup, successful requests, connection timeouts, a database connectivity failure, HTTP 500 errors, login failures, a service restart and critical errors | Log analysis, incident investigation |
| `server_metrics.csv` | 12 rows: 6 servers at two snapshots (13:00 and 14:05). Columns: timestamp, server name, CPU, memory, disk, network latency and status | Capacity planning, calculations |
| `incident_notes.txt` | Incident `INC-20261119-001`: summary, reported time, affected application, user impact, initial observations, actions taken and current status | Incident investigation |

Here we read the files with plain Python so you can see their contents. Later, the agent reads them through the File Management tools.

In [ ]:
# List the sample files and their sizes
for file_path in sorted(WORKSPACE.iterdir()):
    print(f"{file_path.name:22} {file_path.stat().st_size:>6} bytes")

# Show the first 12 lines of application.log
print("\n--- application.log (first 12 lines) ---")
with open(WORKSPACE / "application.log", encoding="utf-8") as log_file:
    for line_number, line in enumerate(log_file, start=1):
        if line_number > 12:
            break
        print(f"{line_number:>3}: {line.rstrip()}")

# The three original sample files. The agent may read them but must never change them.
ORIGINAL_FILES = ["application.log", "server_metrics.csv", "incident_notes.txt"]

# Record a SHA-256 fingerprint of each original file so we can prove later that nothing changed
ORIGINAL_HASHES = {name: hashlib.sha256((WORKSPACE / name).read_bytes()).hexdigest()
                   for name in ORIGINAL_FILES}
print("\nFingerprints recorded for", len(ORIGINAL_HASHES), "original files.")

In [ ]:
# Read server_metrics.csv into a list of dictionaries (one dictionary per row)
with open(WORKSPACE / "server_metrics.csv", newline="", encoding="utf-8") as csv_file:
    metric_rows = list(csv.DictReader(csv_file))

print(f"server_metrics.csv: {len(metric_rows)} rows. First 3 rows:")
for row in metric_rows[:3]:
    print(row)

# Show incident_notes.txt in full
print("\n--- incident_notes.txt ---")
print((WORKSPACE / "incident_notes.txt").read_text(encoding="utf-8"))

## Section 4 - Understand the File Management Tools

LangChain's **File Management toolkit** gives an agent ready-made file tools. Each tool has a **name**, a **description** and a list of **inputs**. The agent sees the name and the description when it chooses a tool.

| Tool | Purpose | Used in this lab? |
|---|---|---|
| `list_directory` | Lists the files and folders in a directory inside the workspace | Yes |
| `read_file` | Reads the text of one file, such as `application.log` | Yes |
| `file_search` | Finds files by name pattern, such as `*.log`. It searches file **names**, not the text inside the files | Yes |
| `write_file` | Creates a new file (or overwrites one) inside the workspace | Yes, with a safety wrapper |
| `copy_file`, `move_file`, `file_delete` | Copy, move or delete files | **No**. Deletion and changes to source files are not allowed |

**Safety controls in this lab**

- Every tool is limited to `operations_workspace/`. The built-in validation rejects paths such as `../secret.txt`.
- `write_file` is wrapped in `SafeWriteFileTool`. It refuses to overwrite the three original sample files, in any letter case and through `..` paths, and accepts only `.txt` or `.md` files. Reports are always new files, so the sources are never changed.
- The copy, move and delete tools are not registered, so the agent cannot use them.

In [ ]:
# Only report files may be written, and the three original sample files may never be overwritten
ALLOWED_REPORT_EXTENSIONS = {".txt", ".md"}


def is_original_file(file_path):
    """True if the path points at an original sample file. The check ignores letter case and
    resolves '..' and symlinks, so 'INCIDENT_NOTES.TXT' or 'sub/../incident_notes.txt' are caught too."""
    target = os.path.normcase(str((WORKSPACE / file_path).resolve()))
    originals = {os.path.normcase(str((WORKSPACE / name).resolve())) for name in ORIGINAL_FILES}
    return target in originals


class SafeWriteFileTool(WriteFileTool):
    """The built-in write_file tool with two lab rules added."""

    def _run(self, file_path, text, append=False, run_manager=None):
        if is_original_file(file_path):
            return f"Error: {Path(file_path).name} is an original source file and cannot be modified. Write a new report file instead."
        if Path(file_path).suffix.lower() not in ALLOWED_REPORT_EXTENSIONS:
            return "Error: only .txt or .md report files can be written."
        return super()._run(file_path, text, append=append, run_manager=run_manager)


# Read-only tools from the toolkit. copy, move and delete are deliberately NOT selected.
read_only_tools = FileManagementToolkit(
    root_dir=str(WORKSPACE),
    selected_tools=["list_directory", "read_file", "file_search"],
).get_tools()

# Add the guarded write tool. root_dir keeps every path inside the workspace.
file_tools = read_only_tools + [SafeWriteFileTool(root_dir=str(WORKSPACE))]

print("File Management tools registered for the agent:")
for tool in file_tools:
    print(f" - {tool.name:15} inputs: {list(tool.args.keys())}")

In [ ]:
tools_by_name = {tool.name: tool for tool in file_tools}

print("== list_directory ('.') ==")
print(tools_by_name["list_directory"].invoke({"dir_path": "."}))

print("\n== file_search (pattern *.log) ==")
print(tools_by_name["file_search"].invoke({"dir_path": ".", "pattern": "*.log"}))

print("\n== read_file: application.log (first 300 characters) ==")
log_text = tools_by_name["read_file"].invoke({"file_path": "application.log"})
print(log_text[:300], "...")
print("Total lines:", log_text.count("\n"))

print("\n== read_file: incident_notes.txt (first 200 characters) ==")
print(tools_by_name["read_file"].invoke({"file_path": "incident_notes.txt"})[:200], "...")

In [ ]:
print("== write_file: create a test report (allowed) ==")
print(tools_by_name["write_file"].invoke({
    "file_path": "lab_test_report.txt",
    "text": "Lab test report\nFile Management tools are working.\n",
}))

print("\n== write_file: try to overwrite application.log (blocked) ==")
print(tools_by_name["write_file"].invoke({"file_path": "application.log", "text": "overwritten"}))

print("\n== read_file: try to read outside the workspace (blocked) ==")
print(tools_by_name["read_file"].invoke({"file_path": "../../secret.txt"}))

print("\n== Are the original files unchanged? ==")
for name in ORIGINAL_FILES:
    current = hashlib.sha256((WORKSPACE / name).read_bytes()).hexdigest()
    print(f"{name:22} unchanged: {current == ORIGINAL_HASHES[name]}")

## Section 5 - Understand the Python REPL Tool

**What is a Python REPL?** A REPL (Read-Eval-Print Loop) runs a piece of code and returns what it prints. LangChain's `PythonREPLTool` gives an agent this ability. The agent writes Python code as text, the tool runs it, and the printed output comes back as the observation.

**How an agent uses it.** For a question such as "What is the average CPU?", the agent usually does two things. It reads `server_metrics.csv` with a File Management tool. Then it writes Python code that contains those numbers and sends it to the REPL, which returns the result. The LLM does not do the arithmetic itself; the program does.

**Direct vs. agent-selected execution.** In the next cells *you* write the Python code and call the tool directly. From Section 7 onwards, *the agent* writes the code after it has chosen this tool.

**Why the REPL must be restricted.** `PythonREPLTool` runs whatever Python code it receives. Without limits, an agent (or a badly written prompt) could delete files or start programs. This lab limits the REPL in three ways:

- Its built-in functions are reduced to a short list of safe ones, such as `print`, `sum`, `max` and `round`.
- Before the code runs, it is checked. Code with `import` statements, or with names that start with double underscores (`__class__`, `__import__`), is rejected. Those are the usual ways out of a restricted namespace.
- Code that runs for more than 5 seconds is stopped, so an endless loop cannot hang the notebook.
- There is no `open` and no access to the operating system, and the REPL cannot read the workspace. The data reaches it only as values written into the code.

This is a **teaching guardrail, not a security sandbox**. A determined person can still find ways around a check like this. In production, untrusted code must run in an isolated container or service with no file or network access.

In [ ]:
import ast
from langchain_experimental.tools.python.tool import sanitize_input

# Only these built-in functions are available inside the REPL
SAFE_BUILTINS = {name: getattr(builtins, name) for name in [
    "abs", "all", "any", "bool", "dict", "divmod", "enumerate", "filter", "float", "int", "isinstance",
    "len", "list", "map", "max", "min", "pow", "print", "range", "reversed", "round", "set", "sorted",
    "str", "sum", "tuple", "zip",
]}


def check_code_is_allowed(code):
    """Reject code that imports modules or uses double-underscore names.
    Those are the usual ways out of a restricted namespace. This is a teaching check, not a sandbox."""
    for node in ast.walk(ast.parse(code)):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            raise PermissionError("imports are not allowed in the lab REPL")
        if isinstance(node, ast.Name) and node.id.startswith("__"):
            raise PermissionError(f"name '{node.id}' is not allowed")
        if isinstance(node, ast.Attribute) and node.attr.startswith("__"):
            raise PermissionError(f"attribute '{node.attr}' is not allowed")


REPL_TIMEOUT_SECONDS = 5  # code that runs longer than this is stopped


class GuardedPythonREPLTool(PythonREPLTool):
    """PythonREPLTool that checks the code before it runs and stops code that runs too long."""

    def _run(self, query, run_manager=None):
        code = sanitize_input(query)
        try:
            check_code_is_allowed(code)
        except SyntaxError as error:
            return f"Error: SyntaxError: {error.msg} (line {error.lineno})"
        except PermissionError as error:
            return f"Error: {error}"
        return self.python_repl.run(code, timeout=REPL_TIMEOUT_SECONDS)


# PythonREPLTool with a restricted namespace: its globals contain only SAFE_BUILTINS
python_repl_tool = GuardedPythonREPLTool(
    python_repl=PythonREPL(_globals={"__builtins__": SAFE_BUILTINS})
)
print("Tool name:", python_repl_tool.name)

print("\n== Blocked: importing the os module ==")
print(python_repl_tool.invoke("import os\nprint(os.listdir('.'))"))

print("\n== Blocked: opening a file ==")
print(python_repl_tool.invoke("print(open('application.log').read())"))

print("\n== Blocked: double-underscore names (used to escape a restricted namespace) ==")
print(python_repl_tool.invoke("print(().__class__)"))

print(f"\n== Stopped: code that runs longer than {REPL_TIMEOUT_SECONDS} seconds ==")
print(python_repl_tool.invoke("while True:\n    pass"))

In [ ]:
# 1. Basic calculation: hours in a week, divided by 12
print("Basic calculation:", python_repl_tool.invoke("print(round(7 * 24 / 12, 2))"))

# 2. Average CPU utilization. The values come from server_metrics.csv (read in Section 3).
cpu_values = [float(row["cpu_utilization"]) for row in metric_rows]
cpu_code = (f"cpu = {cpu_values}\n"
            "print('Average CPU %:', round(sum(cpu) / len(cpu), 2))\n"
            "print('Samples:', len(cpu))")
print("\nAverage CPU:")
print(python_repl_tool.invoke(cpu_code))

# 3. Highest disk utilization: which server and which snapshot?
disk_values = [(row["server_name"], row["timestamp"], float(row["disk_utilization"])) for row in metric_rows]
disk_code = (f"disks = {disk_values}\n"
             "top = max(disks, key=lambda item: item[2])\n"
             "print('Highest disk (server, time, %):', top)")
print("Highest disk:")
print(python_repl_tool.invoke(disk_code))

# 4. Simple capacity calculation: free space on an example 500 GB volume that is 93.8% full
print("Capacity example:")
print(python_repl_tool.invoke("print('Free space GB:', round(500 * (100 - 93.8) / 100, 1))"))

## Section 6 - Understand the Web/Search Tool

**What do Web/Search tools do?** They send a query to a public search engine and return the results (titles, snippets and links). They give the agent knowledge that is not in the workspace, such as generic troubleshooting advice.

**Why might an agent need it?** The sample files show *what happened here*. A search shows *what is commonly done* for a type of problem, such as high CPU. These are different kinds of information and must not be mixed up.

**How do results become observations?** The search tool returns text, and the agent reads it as an observation, just like the output of a file tool. The text comes from outside and is untrusted. It can be incomplete or out of date, so the agent must label it as reference information.

| | Internal data | External information |
|---|---|---|
| Source | Files in `operations_workspace/` | Public web search (DuckDuckGo) |
| Example | "app-srv-02 CPU was 96.4% at 14:05" | "Common causes of high CPU include runaway processes..." |
| Can it be sent to the search provider? | **Never** | Only generic, public questions |
| How to treat it | Facts to cite | Reference to check, not verified fact |

**Tool used:** `DuckDuckGoSearchRun` from `langchain-community`. It needs no API key. It uses the `ddgs` package to query DuckDuckGo.

In [ ]:
search_tool = DuckDuckGoSearchRun()
print("Search tool name:", search_tool.name)

# A generic public question: no hostnames, IP addresses, log lines or incident IDs
query = "common causes of high server CPU utilization"
results = search_tool.invoke({"query": query})

print(f"\nQuery sent: '{query}'")
print(f"Characters returned: {len(results)}")
print("\nFirst 500 characters of the results (external reference, not verified):")
print(results[:500])

## Section 7 - Build the LangChain Agent

This is the main section of the lab.

**What makes up an agent in code?** Three parts work together:

1. **An LLM** (`ChatOpenAI`, model `gpt-4o-mini`). It reads the request and decides what to do.
2. **Tools.** These are the functions the LLM is allowed to call. We register exactly three categories: File Management, Python REPL and Web/Search.
3. **A prompt.** The system prompt gives the LLM its role and rules. A placeholder (`agent_scratchpad`) records each tool call and observation during the run.

**How are tools registered?** Each tool has a `name`, a `description` and typed inputs. LangChain sends these descriptions to the LLM with every request. The LLM chooses a tool by matching the request against the descriptions, so clear descriptions lead to better tool choices.

**How does the agent decide?** `create_tool_calling_agent` lets the LLM return either a tool call or a final answer. `AgentExecutor` runs the loop. It executes the requested tool, sends the observation back to the LLM, and repeats until there is a final answer or `MAX_ITERATIONS` is reached. `verbose=True` prints every step so you can follow the reasoning.

**Why `langchain_classic`?** In LangChain 1.x, `create_tool_calling_agent` and `AgentExecutor` moved to the `langchain-classic` package, which is installed together with `langchain-community` 0.4.2. The newer `create_agent` API is built on LangGraph, which is outside this lab's scope, so it is not used.

**Tool error handling.** Every tool has `handle_tool_error = True`. If a tool fails, the error text goes back to the LLM as an observation, and the agent can try another approach. The notebook does not stop.

**What the system prompt asks for.** Act as an IT Operations assistant. Choose tools that match the request. Use File Management for files, the Python REPL for calculations, and Web/Search only for generic public questions. Separate observed facts from assumptions. Never invent log entries or metrics. Say when information is missing. Never send internal data to the search tool. Save reports only inside the workspace.

In [ ]:
SYSTEM_PROMPT = """You are an IT Infrastructure Operations Assistant for an operations team.

Your job:
- Understand the operational request.
- Choose the most suitable tools. Use several tools when the task needs them.
- File Management (list_directory, file_search, read_file, write_file): inspect operational files in the workspace.
- Python REPL (Python_REPL): run calculations and counts. Write the code yourself and print the results.
- Web/Search (duckduckgo_search): look up generic public technical guidance only.

Rules:
- Base every finding on tool output. Never invent log entries, metrics, timestamps or servers.
- Clearly separate OBSERVED FACTS (from the files) from ASSUMPTIONS and from EXTERNAL reference guidance.
- If the required information is not available, say so.
- Never send internal data to the search tool: no log lines, server names, IP addresses or incident IDs.
- Only write report files inside the workspace, as .txt or .md. Never overwrite the original sample files.
- Only say that an action succeeded if the tool reported success.
- Give a short, structured answer."""

# Register exactly the three tool categories
ALL_TOOLS = file_tools + [python_repl_tool, search_tool]

# Let the agent recover from tool errors: the error text becomes an observation
for tool in ALL_TOOLS:
    tool.handle_tool_error = True

print("Registered tools (3 categories):")
for tool in ALL_TOOLS:
    print(f" - {tool.name}")

In [ ]:
MISSING_KEY_MESSAGE = "OPENAI_API_KEY is missing. Add it to .env (see the README) and restart the kernel."
if not api_key_found:
    # Every agent cell calls run_agent, so give it the same clear message instead of a NameError
    def run_agent(request):
        raise EnvironmentError(MISSING_KEY_MESSAGE)

    raise EnvironmentError(MISSING_KEY_MESSAGE)

# 1. The LLM
llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)

# 2. The prompt: system rules, the user request, and the scratchpad of tool steps
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

# 3. The agent (decides which tool to call) and the executor (runs the loop)
agent = create_tool_calling_agent(llm, ALL_TOOLS, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=ALL_TOOLS,
    verbose=True,                    # print every step for the classroom
    max_iterations=MAX_ITERATIONS,   # safety limit on the loop
    return_intermediate_steps=True,  # keep the tool trace
    handle_parsing_errors=True,
)


def run_agent(request):
    """Send one request to the agent, then print the tools it used and its answer."""
    result = agent_executor.invoke({"input": request})
    tools_used = [step[0].tool for step in result["intermediate_steps"]]
    print("\n=== TOOLS USED (in order) ===")
    print(tools_used if tools_used else "No tools were used.")
    print("\n=== AGENT RESPONSE ===")
    print(result["output"])
    return result


print("Agent ready. Model:", CHAT_MODEL, "| max steps:", MAX_ITERATIONS)

## Section 8 - Observe Agent Tool Selection

Now send requests **without telling the agent which tool to use**. In the verbose trace, each `Invoking:` line shows the tool the agent chose and the input it wrote. The `TOOLS USED` summary lists the tools in order.

| Request | Expected tool(s) | Why |
|---|---|---|
| List the files available in the Operations workspace | File Management (`list_directory`) | The request is about files |
| Calculate the average CPU utilization of all servers | File Management (`read_file`), then Python REPL | The data is in a CSV file and the calculation needs Python. The REPL cannot read files itself |
| Search for common causes of high server CPU utilization | Web/Search (`duckduckgo_search`) | Generic public knowledge, not in the workspace |
| Read the application logs, identify critical errors and prepare a summary | File Management (`read_file`), plus Python REPL if counting is needed | The log content comes first, then the counting |

The agent may take a different but reasonable path. The key test is whether it used a tool to get the facts, instead of answering from memory.

In [ ]:
print("########## Example 1: File Management ##########")
run_agent("List the files available in the Operations workspace.")

print("\n\n########## Example 2: calculation ##########")
run_agent("Calculate the average CPU utilization of all servers.")

In [ ]:
print("########## Example 3: Web/Search ##########")
run_agent("Search for common causes of high server CPU utilization.")

print("\n\n########## Example 4: logs and summary ##########")
run_agent("Read the application logs, identify critical errors and prepare a summary.")

## Section 9 - Practical IT Operations Use Cases

Each use case is one request, and the agent must combine tools to answer it. Compare the `TOOLS USED` list with the expected workflow.

**Sample thresholds used in this lab.** These are training values, **not** universal production policies:

- CPU above 85%: **high** utilization.
- Memory above 85%: **high** utilization.
- Disk above 90%: **critical** utilization.
- Network latency above 100 ms: **elevated** (an extra sample threshold for this lab).

| Use case | Expected workflow | Tools |
|---|---|---|
| 1. Log analysis | Read `application.log`, identify the errors, count the critical errors, summarise | File Management, Python REPL (counting) |
| 2. Capacity planning | Read `server_metrics.csv`, calculate, compare with thresholds, rank servers, recommend | File Management, Python REPL |
| 3. Incident investigation | Read `incident_notes.txt`, read `application.log`, correlate timestamps, separate evidence from assumptions, suggest steps. Web/Search only if generic guidance is needed | File Management, optional Web/Search |
| 4. Operational reporting | Read logs and metrics, calculate, summarise risks, save `daily_operations_report.txt` | File Management, Python REPL, File Management (write) |

In [ ]:
# USE CASE 1 - LOG ANALYSIS
run_agent(
    "Analyze the application logs, identify the major errors, count the critical errors "
    "and prepare an incident summary."
)

In [ ]:
# USE CASE 2 - CAPACITY PLANNING (sample thresholds are stated in the request)
run_agent(
    "Analyze server_metrics.csv and identify servers with high CPU, memory and disk utilization. "
    "Use these sample thresholds: CPU above 85% and memory above 85% are high; disk above 90% is critical; "
    "network latency above 100 ms is elevated. Rank the servers by operational concern and recommend "
    "which servers need immediate attention."
)

In [ ]:
# USE CASE 3 - INCIDENT INVESTIGATION
run_agent(
    "Investigate the incident described in incident_notes.txt. Review the application logs, "
    "identify related errors and suggest troubleshooting steps. Clearly separate evidence from assumptions."
)

In [ ]:
# USE CASE 4 - OPERATIONAL REPORTING (the agent saves a new report file)
run_agent(
    "Prepare a daily IT Operations report covering application errors, server capacity and key "
    "operational risks. Save it as daily_operations_report.txt in the operations workspace. "
    "The report must contain these sections: Executive summary; Major application errors; "
    "Server capacity observations; Critical operational risks; Recommended actions."
)

In [ ]:
report_path = WORKSPACE / "daily_operations_report.txt"
print("Report saved:", report_path.exists())

if report_path.exists():
    report_text = report_path.read_text(encoding="utf-8")
    required_sections = ["Executive summary", "Major application errors", "Server capacity",
                         "Critical operational risks", "Recommended actions"]
    print("Required sections found:", [s for s in required_sections if s.lower() in report_text.lower()])
    print("Report length (characters):", len(report_text))
    print("\n--- first 600 characters ---")
    print(report_text[:600])

# Confirm that no agent run changed the three original sample files
print("\nOriginal files unchanged:")
for name in ORIGINAL_FILES:
    current = hashlib.sha256((WORKSPACE / name).read_bytes()).hexdigest()
    print(f" - {name:22} {'OK' if current == ORIGINAL_HASHES[name] else 'CHANGED!'}")

## Section 10 - Understanding Agent Execution

Here is the sequence for one request. The example is Use Case 4: *"Prepare a daily IT Operations report ... save it as daily_operations_report.txt."*

| Step | What happens | Where you see it |
|---|---|---|
| 1. User request | You pass the text to `run_agent(...)` | The input of the executor |
| 2. Agent interpretation | The LLM reads the request and the tool descriptions. It works out that it needs logs, metrics, a calculation and a saved file | Its next tool choice |
| 3. Tool selection | It chooses `read_file` for `application.log` | `Invoking: read_file ...` |
| 4. Tool execution | LangChain runs the tool, limited to the workspace | The tool output in the trace |
| 5. Tool observation | The file text is returned to the agent | The output printed under the invoke line |
| 6. Further tool selection | It reads `server_metrics.csv`, runs a calculation in the Python REPL, then calls `write_file` | More `Invoking:` lines |
| 7. Final response | The agent summarises what it observed and says where the report was saved | `=== AGENT RESPONSE ===` |

**Key points**

- An agent may call **one tool or several**, depending on the task.
- It does **not** call every available tool for every request. Example 1 needs no Python REPL, and Use Case 1 needs no web search.
- The final answer should rest on the observations. If a tool returned an error, the agent should report it rather than hide it.
- Every `Invoking:` line is a real tool execution. The tools are LangChain's built-in tools, not ordinary Python functions that imitate them.

## Section 11 - Hands-On Exercises and Quiz

**Exercises.** Add a new code cell below for each exercise and call `run_agent("...")` with the request. Check the `TOOLS USED` list every time.

1. **List files:** "List all files in the Operations workspace."
2. **Highest CPU:** "Identify the server with the highest CPU utilization."
3. **Critical errors:** "Identify and count the critical application errors in application.log."
4. **Incident investigation:** "Investigate incident INC-20261119-001 using incident_notes.txt and application.log."
5. **Multi-tool report:** "Prepare an operational report covering application errors, server capacity and risks. Save it as my_report.txt in the workspace."

**Quiz.** Choose one answer for each question.

1. What best describes the difference between a normal LLM application and an AI agent?
   - a) An agent never uses a language model.
   - b) An agent can choose tools, run them and use their outputs before it answers.
   - c) An agent always answers faster.
   - d) An agent needs a database to work.

2. The request is "Calculate the average memory utilization from server_metrics.csv". Which combination is most suitable?
   - a) Web/Search only.
   - b) Python REPL only, with direct file access.
   - c) File Management to read the CSV, then the Python REPL to calculate.
   - d) No tool, because the LLM can calculate the average itself.

3. Why is `file_delete` not registered for the agent in this lab?
   - a) Deletion is slow.
   - b) Deleting files is a destructive action that this lab does not allow.
   - c) LangChain does not support file deletion.
   - d) The workspace is read-only.

4. Why are the Python REPL's built-in functions restricted in this lab?
   - a) To make calculations faster.
   - b) Because `PythonREPLTool` runs arbitrary code, so imports, file access and system commands must be blocked.
   - c) Because Python cannot calculate averages with the full set of built-ins.
   - d) To reduce the number of tokens sent to the LLM.

5. What is a tool observation?
   - a) The LLM's guess about what a tool would return.
   - b) The output a tool returned, which the agent reads before its next step or final answer.
   - c) A log entry written by the OpenAI API.
   - d) The Mermaid diagram in Section 1.

## Instructor Answer Key

*For the facilitator only.*

**Quiz answers**

1. **b.** The agent chooses tools, runs them and uses the results before it answers. A normal LLM application answers from the prompt and its training data.
2. **c.** The data is in a file, so the agent must read it first. The REPL does the arithmetic, so the LLM does not guess the result.
3. **b.** `file_delete` is excluded on purpose. The lab forbids deletions and changes to source files.
4. **b.** The REPL executes any Python code it receives. Restricting the built-ins blocks imports, file access and system commands. It is a teaching guardrail, not a full sandbox.
5. **b.** The observation is the tool's real output. The agent uses it for its next decision or its final answer.

**Expected results from the sample data** (computed from the files; participants' wording will vary)

- **Log analysis:** 32 log lines. 3 CRITICAL, 8 ERROR, 4 WARN and 17 INFO. 5 HTTP 500 responses (all `DB_CONNECTION_TIMEOUT`), 2 failed logins (`jsmith`, `mkumar`), and the first connection timeout at 14:02:11.
- **Highest CPU:** `app-srv-02` at 14:05 (96.4%).
- **Averages (all 12 rows):** CPU 56.87%, memory 70.60%, disk 69.44%.
- **Highest disk:** `batch-srv-01` at 14:05 (93.8%). It was also critical at 13:00 (92.6%).
- **High CPU (above 85%):** `app-srv-01` (88.9%) and `app-srv-02` (96.4%), both at 14:05.
- **High memory (above 85%):** `db-srv-01` (88.2% at 13:00 and 89.5% at 14:05) and `app-srv-02` (91.7% at 14:05).
- **Elevated latency (above 100 ms):** `web-srv-02` (115.7 ms at 14:05).
- **Incident investigation:** a good answer links the 14:02 timeouts, the 14:02:26 connection refusal, the 14:05 pool exhaustion and the 14:20 restart. It states that the root cause is **not confirmed**, and lists the open actions from the notes. A weak answer presents a root cause as fact.
- **Report:** `daily_operations_report.txt` is created in the workspace and the three original files are unchanged.

**Limitations to discuss with the class**

- The LLM can copy numbers wrongly into the REPL code. Ask the agent to show its code and check it.
- Search results are external and untrusted. They are not evidence about this system.
- `file_search` matches file names only. Finding text inside a file depends on `read_file` plus the LLM's reading.
- The REPL guardrail is a teaching control, not a security sandbox.
- Agent answers vary between runs. Tool choices can differ, but the facts should match the files.